遍历 fetal_hydro_dataset_2\raw 下所有病例
找到每一次检查（如 202412300188 这种目录）
在该目录下创建 left/ 和 right/ 两个文件夹（与 meta.json / report.txt / images 同级）
如果已经存在则跳过，不会报错

In [2]:
import os
from pathlib import Path

# 根目录（按需修改）
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

created_count = 0
skipped_count = 0

for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for phase_dir in case_dir.iterdir():  # 1prenatal / 2postnatal / 3postoperative
        if not phase_dir.is_dir():
            continue

        for visit_dir in phase_dir.iterdir():  # 每一次检查
            if not visit_dir.is_dir():
                continue

            # 可选：简单判断是不是检查目录（包含 images 或 meta.json）
            if not (visit_dir / "images").exists() and not (visit_dir / "meta.json").exists():
                continue

            left_dir = visit_dir / "left"
            right_dir = visit_dir / "right"

            try:
                if not left_dir.exists():
                    left_dir.mkdir()
                if not right_dir.exists():
                    right_dir.mkdir()

                created_count += 1

            except Exception as e:
                print(f"创建失败: {visit_dir}, 错误: {e}")
                skipped_count += 1

print(f"处理完成: {created_count} 个检查目录")
print(f"失败/跳过: {skipped_count} 个")

处理完成: 205 个检查目录
失败/跳过: 0 个


下面这段代码直接在刚刚创建好的 left/ 和 right/ 下面：

创建 images/ 文件夹
创建空的 report.txt
已存在则跳过（安全可重复运行）

In [3]:
import os
from pathlib import Path

# 根目录
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

created_count = 0
skipped_count = 0

for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for phase_dir in case_dir.iterdir():
        if not phase_dir.is_dir():
            continue

        for visit_dir in phase_dir.iterdir():
            if not visit_dir.is_dir():
                continue

            # 只处理有 left/right 的目录
            left_dir = visit_dir / "left"
            right_dir = visit_dir / "right"

            for side_dir in [left_dir, right_dir]:
                if not side_dir.exists():
                    continue

                try:
                    # 创建 images 文件夹
                    images_dir = side_dir / "images"
                    if not images_dir.exists():
                        images_dir.mkdir()

                    # 创建空 report.txt
                    report_path = side_dir / "report.txt"
                    if not report_path.exists():
                        report_path.touch()

                    created_count += 1

                except Exception as e:
                    print(f"处理失败: {side_dir}, 错误: {e}")
                    skipped_count += 1

print(f"处理完成: {created_count} 个 side 目录")
print(f"失败/跳过: {skipped_count} 个")

处理完成: 410 个 side 目录
失败/跳过: 0 个


下面的代码就是把原始 report.txt 同步复制到 left/right下的report.txt，方便后面人工删减。

遍历所有检查目录
读取当前 report.txt
写入到 left/report.txt 和 right/report.txt
如果原始 report 不存在则跳过

In [4]:
import shutil
from pathlib import Path

# 根目录
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

processed_count = 0
skipped_count = 0

for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for phase_dir in case_dir.iterdir():
        if not phase_dir.is_dir():
            continue

        for visit_dir in phase_dir.iterdir():
            if not visit_dir.is_dir():
                continue

            # 原始 report
            src_report = visit_dir / "report.txt"

            if not src_report.exists():
                skipped_count += 1
                continue

            # 左右路径
            left_report = visit_dir / "left" / "report.txt"
            right_report = visit_dir / "right" / "report.txt"

            try:
                # 确保目录存在（保险）
                left_report.parent.mkdir(parents=True, exist_ok=True)
                right_report.parent.mkdir(parents=True, exist_ok=True)

                # 复制内容（覆盖写）
                shutil.copyfile(src_report, left_report)
                shutil.copyfile(src_report, right_report)

                processed_count += 1

            except Exception as e:
                print(f"处理失败: {visit_dir}, 错误: {e}")
                skipped_count += 1

print(f"处理完成: {processed_count} 个检查")
print(f"跳过: {skipped_count} 个")

处理完成: 205 个检查
跳过: 0 个
